In [1]:
# Auto reload
%load_ext autoreload
%autoreload 2

# library imports
from matplotlib import pyplot as plt
import numpy as np

# import PGL Eyelink interface
from pgl import pglEyelink, pglEyelinkData, pglTask, pglParameter

# Load PGL libraries and start a PGL window
from pgl import pgl, pglExperiment
pgl = pgl()
e = pglExperiment(pgl, settingsName="ViewPixx")

# close any existing windows
pgl.cleanUp()


================================ pglBase: init =================================
(pgl) mglMetal error log can be viewed in MacOS Console app by searching for PROCESS mglMetal or in a terminal with:
      log stream --level info --process mglMetal
(pgl) To search for something specifc, e.g. messages from mglMovie:
      log stream --predicate 'eventMessage CONTAINS "mglMovie"' --style syslog --level info
(pgl:checkOS) Python version: 3.12.12 | packaged by conda-forge | (main, Jan 27 2026, 00:01:15) [Clang 19.1.7 ]
(pgl:checkOS) Running on Mac mini (Mac14,3) with macOS version: 15.6.1
(pgl:checkOS) Apple M2 Cores: 8 (4 performance and 4 efficiency) Memory: 16 GB
(pgl:checkOS) GPU: Apple M2 (Built-In) 10 cores, Metal 3 support
(pgl:checkOS)   HP ZR2440w [Main Display]: 1920 x 1200 (WUXGA - Widescreen Ultra eXtended Graphics Array) (Unknown type) GammaTable size: 1024
(pgl:checkOS)   VIEWPixx3D: 1920 x 1080 (1080p FHD - Full High Definition) (Unknown type) GammaTable size: 1024
(pglBase) M

In [2]:

e.initScreen()
eyelink = pglEyelink(pgl)

================================= pglBase:open =================================
(pglBase:open) Starting mglMetal application: /Users/justin/proj/pgl/metal/mglMetal.app
(pglBase:open) Using socket with address: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260513_171657.K28lrk0l36
(pgl:_pglComm) .Connected to: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260513_171657.K28lrk0l36
(pgl:_resolution:getResolution) Display 1/2: 1920x1080 60Hz 32bits
(pglKeyboardMouse:start) Starting keyboard and mouse event listener.
(pglDevices) Added device: pglKeyboard
(pglEventListener) Eating 7 keys: ['1', '2', '3', '4', '`', 'escape', 'space']
(pglEyelink) Attempting to connect to Eyelink at 100.1.1.1...
displayAPI: Connecting
displayAPI: Disconnect
displayAPI: Connecting
(pglEyelink) Using pgl display for Eyelink calibration and validation.


1495 thread_policy_set failed: 4.
1466 thread_policy_set failed: 4.
1495 thread_policy_set failed: 4.
1495 thread_policy_set failed: 4.
1495 thread_policy_set failed: 4.


In [ ]:
# TODO
# 1 abort calibration key
# *** 2 Eat all keys
# *** 3 setting for eccentricity of calibration
# 4 Check Read / Write EDF
# 5 Send triggerq to EDF 
# 6 Handle more gracefully stopping keyboard - with interrupt handler for eyelink
# 7 Force a
# qQccept of point
# 8 Display crosses on eye screen
# 9 
#eyelink.setCustomCalibrationPoints(margin=0.7, numPoints=9)
#eyelink.calibrate()
#e.endScreen()
eyelink.openEDF('test')
eyelink.start()
pgl.waitSecs(2)
eyelink.sendMessage("Test 1")
pgl.waitSecs(2)
eyelink.sendMessage("Test 2")
pgl.waitSecs(1)
eyelink.stop()
x = eyelink.getEDF('test','/Users/justin/data')


In [ ]:
#d=pglEyelinkData('/Users/justin/Desktop/test.asc')
d=pglEyelinkData('/Users/justin/data/test.asc')

In [ ]:
print(d)
#plt.plot(d.samples['time'],d.samples['x'],'.')
#d.samples
#d.messages
#d.fixations
#d.saccades
d.blinks

In [3]:
# Set up eye fixation task
class pgFixationTask(pglTask):
    
    ########################
    def __init__(self, pgl, eyelink):
        super().__init__(pgl)
        self.eyelink = eyelink
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "Eye Fixation Task"
        self.settings.seglen = [1.5]
        
        # fixed parameters, these will automatically be saved in the settings file
        self.settings.fixedParameters = {
            'nPoints':9,
            'width':10,
            'height':10
        }        
        p = self.settings.fixedParameters

        # compute positions of calibration points
        if p['nPoints']==9:
            x = p['width']*0.5
            y = p['height']*0.5
            self.settings.fixedParameters['calibrationPoints'] = [(0,0),(-x,y),(0,y),(x,y),(-x,0),(x,0),(-x,-y),(0,-y),(x,-y)]
        else:
            raise ValueError(f"Unsupported number of points: {p['nPoints']}")
        
        fixIndex = pglParameter('fixIndex',np.arange(p['nPoints']))
        self.addParameter(fixIndex)
        
    ########################
    def updateScreen(self):
        if self.state.currentSegment==0:
            # get the current fix Index
            fixIndex = self.currentParams['fixIndex']
            # get the position
            x = self.settings.fixedParameters['calibrationPoints'][fixIndex][0]
            y = self.settings.fixedParameters['calibrationPoints'][fixIndex][1]
            self.pgl.circle(0.25,x,y,fill=True)
            self.pgl.fixationCross(0.25,x,y,color=[1,0,0])
    def startTrial(self, startTime):
        # call super
        super().startTrial(startTime)
        # send a message to eyelink
        eyelink.sendMessage(f"pgl: Trial start trialNum={self.state.currentTrial}  timestamp={startTime}")
    def startSegment(self, updateTime):
        # call super
        super().startSegment(updateTime)
        # send a message to eyelink
        eyelink.sendMessage(f"pgl: Segment start segmentNum={self.state.currentSegment} trialNum={self.state.currentTrial} timestamp={updateTime}")

# initialize task
fixTask = pgFixationTask(pgl,eyelink)


In [4]:
e.initScreen()
e.addTask(fixTask)

eyelink.setCustomCalibrationPoints(margin=0.7, numPoints=9)
eyelink.calibrate()

eyelink.openEDF('test')
eyelink.start()
e.run()
eyelink.stop()
eyelink.getEDF('test','/Users/justin/data')

e.endScreen()
e.pgl.close()


================================= pglBase:open =================================
(pglBase:open) Starting mglMetal application: /Users/justin/proj/pgl/metal/mglMetal.app
(pglBase:open) Using socket with address: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260513_171717.83MD3KJGv5
(pgl:_pglComm) .Connected to: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260513_171717.83MD3KJGv5
(pgl:_resolution:getResolution) Display 1/2: 1920x1080 60Hz 32bits
(pglEventListener) Eating 7 keys: ['1', '2', '3', '4', '`', 'escape', 'space']
(pglEyelink:calibrate) Starting calibration routine
             Enter: Show camera image
             C: (C)alibrate V: (V)alidate
             0 or Q: (Q)uit calibration
(pglEventListener) Eating 5 keys: ['0', 'c', 'q', 'return', 'v']
>>> Processing keydown: 'return' (code: 36)
>>> Mapped special key 'return' -> 13
(pglEyelink) setup_image_display
>>> Processing keydown: 'return' (code: 36)
>>> Mapped special key 'retur

True

In [ ]:
eyelink.openEDF('test')
eyelink.start()
pgl.waitSecs(2)
eyelink.sendMessage("Test 1")
pgl.waitSecs(2)
eyelink.sendMessage("Test 2")
pgl.waitSecs(1)
eyelink.stop()
x = eyelink.getEDF('test','/Users/justin/data')

In [ ]:
help(pgl.circle)

In [ ]:
#from pgl import pglEyelink
pgl.cleanUp()
pgl.open(0,800,600)
pgl.visualAngle(57,30,20)
from pgl import pglEyelinkCustomDisplay
#eyelink = pglEyelink(pgl)
#customDisplay = pglEyelinkCustomDisplay(pgl,eyelink)

customDisplay = pglEyelinkCustomDisplay(pgl)

customDisplay.clear_cal_display()
customDisplay.draw_line(00,0,500,500,0)
customDisplay.draw_lozenge(100,200,100,300,0)
customDisplay.draw_cal_target(300,300)
#customDisplay.get_input_key()
#pgl.flush()

In [ ]:
eyelink.close()

In [ ]:
import pylink

def setup_eye_link(eyelink_address="100.1.1.1"):
    try:
        # Create an EyeLink tracker object.
        el_tracker = pylink.EyeLink(eyelink_address)
        print(f"Connecting to EyeLink at {eyelink_address}...")
        el_tracker.open()  # Open the connection to the EyeLink
        print("EyeLink connected.")

        # Check connection status
        if el_tracker.isConnected():
            print("EyeLink is connected.")
        else:
            print("EyeLink is not connected.")
            return

        # Setup and calibrate the tracker
        print("Performing tracker setup...")
        el_tracker.doTrackerSetup()  # Setup the tracker (calibration and validation)
        print("Tracker setup completed.")

        # Optionally, you could start recording data here
        # el_tracker.startRecording(1, 1, 1, 1)  # Start recording

        # Close the connection when finished
        el_tracker.close()
        print("EyeLink connection closed.")

    except pylink.EyeLinkError as e:
        print(f"An EyeLink error occurred: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Run the setup
setup_eye_link()